# GenAI Motor Insurance Pricing Pipeline Analysis
This notebook runs the full pipeline for data ingestion, feature engineering, modeling, fraud detection, and scenario simulation for urban traffic conditions.

In [1]:
import sys
sys.path.append('../src')
from data_loader import load_data, preprocess_data
from feature_engineering import create_features
from frequency_model import train_frequency_model, predict_frequency
from severity_model import train_severity_model, predict_severity
from synthetic_data_generator import generate_synthetic_data
from fraud_detection import inject_fraud, detect_anomalies
from pricing_engine import calculate_premium
from simulation import simulate_urban_traffic
from visualization import plot_risk_distribution, plot_premium_distribution, plot_scenario_comparison, plot_fraud_anomalies, generate_reports

In [2]:
# 1. Load and Preprocess Data
df_raw = load_data('../data/drivers.csv')
df = preprocess_data(df_raw)

# 2. Feature Engineering
df_features = create_features(df)
df_features.head()

Data loaded successfully. Shape: (100, 9)
Data preprocessing completed.
Feature engineering completed.


,driver_id,age,vehicle_age,daily_mileage,night_driving_level,harsh_braking_level,accidents_last_2yr,claim_history,night_driving_encoded,harsh_braking_encoded,...,vehicle_type_SUV,vehicle_type_Sedan,vehicle_type_Truck,annual_mileage,annual_mileage_normalized,braking_score,night_driving_score,accident_history_score,driver_risk_index,risk_category
0,D001,56,19,97,Low,High,0,0,1,3,...,0,1,0,35405,0.638298,1.000000,0.333333,0.000000,0.592908,Medium
1,D002,25,6,126,Low,Low,0,0,1,1,...,0,0,0,45990,0.843972,0.333333,0.333333,0.000000,0.410993,Low
2,D003,28,10,92,High,Low,0,0,3,1,...,1,0,0,33580,0.602837,0.333333,1.000000,0.000000,0.517376,Medium
3,D004,57,2,57,High,Medium,2,0,3,2,...,0,1,0,20805,0.354610,0.666667,1.000000,0.666667,0.671986,High
4,D005,38,0,62,Low,Low,0,0,1,1,...,0,1,0,22630,0.390071,0.333333,0.333333,0.000000,0.297518,Low


In [3]:
# 3. Model Training & Prediction
freq_model = train_frequency_model(df_features)
sev_model = train_severity_model(df_features)

df_features['expected_frequency'] = predict_frequency(freq_model, df_features)
df_features['expected_severity'] = predict_severity(sev_model, df_features)

Training Accident Frequency Model (Poisson regression)...
                 Generalized Linear Model Regression Results                  
Dep. Variable:       target_frequency   No. Observations:                  100
Model:                            GLM   Df Residuals:                       94
Model Family:                 Poisson   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -49.151
Date:                Wed, 18 Mar 2026   Deviance:                       52.016
Time:                        00:46:32   Pearson chi2:                     61.5
No. Iterations:                     5   Pseudo R-squ. (CS):           0.009978
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------

In [4]:
# 4. Calculate Premiums
df_premium = calculate_premium(df_features, df_features['expected_frequency'], df_features['expected_severity'])
df_premium[['driver_id', 'risk_category', 'final_premium']].head()

,driver_id,risk_category,final_premium
0,D001,Medium,9927.234411
1,D002,Low,8518.135891
2,D003,Medium,8595.538079
3,D004,High,13385.344613
4,D005,Low,9152.072458


In [5]:
# 5. Visualizations
plot_risk_distribution(df_premium)
plot_premium_distribution(df_premium)
generate_reports(df_premium)
print("Plots generated in results/plots")

Saved premium report to C:\Users\BIT\Desktop\research 101\genai-insurance-pricing\results\premium_reports\top_premiums.csv
Plots generated in results/plots


In [6]:
# 6. Scenario Simulation (Jaipur Traffic)
df_sim = simulate_urban_traffic(df_features, scenario_name="Jaipur High Congestion")

# Recalculate predictions and premiums for simulation
df_sim['expected_frequency'] = predict_frequency(freq_model, df_sim)
df_sim['expected_severity'] = predict_severity(sev_model, df_sim)
df_sim_premium = calculate_premium(df_sim, df_sim['expected_frequency'], df_sim['expected_severity'])

plot_scenario_comparison(df_premium, df_sim_premium, metric='final_premium')
print("Scenario simulation completed. Comparison plot generated.")

Applying scenario constraints: Jaipur High Congestion
Scenario simulation completed. Comparison plot generated.


In [7]:
# 7. Fraud Detection
df_fraud = inject_fraud(df_premium)
df_detected, model = detect_anomalies(df_fraud)

frauds = df_detected[df_detected['anomaly_flag'] == 1]
print(f"Detected {len(frauds)} anomalies/frauds.")
if len(frauds) > 0:
    plot_fraud_anomalies(df_detected)

Running anomaly detection (Isolation Forest)...


Detected 2 anomalies/frauds.
